# 02 — O grafo do schema

Visualiza a estrutura relacional que o RelBench recebe.

A diferença em relação à versão anterior: ela tinha as arestas **escritas à mão**
numa lista de tuplas, copiadas do dicionário de dados. Duas listas paralelas
divergem, e essa divergiu — foi o que motivou D-05. Agora as arestas vêm de
`src.schema.CNES_FKEY`, derivado de `docs/01-selecao-tabelas.md`, então o desenho
mostra o grafo que o código realmente monta.

In [ ]:
import sys
from pathlib import Path

BASE_DIR = Path.cwd().parent
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd

from src import schema
from src.graph import TABELA_RAIZ

G = nx.DiGraph()
for tabela in schema.FACT_TABLES:
    G.add_node(tabela)
for origem, fkeys in schema.CNES_FKEY.items():
    for coluna, destino in fkeys.items():
        G.add_edge(origem, destino, coluna=coluna)

print(f"nós: {G.number_of_nodes()}  arestas: {G.number_of_edges()}")
print(f"tabelas fora do escopo (não desenhadas): {len(schema.TABELAS_FORA)}")


## Quem liga em quê

`tbEstabelecimento` é a raiz. O grau de entrada dela mede quantas tabelas de fato
penduram no estabelecimento — é o que dá à GNN heterogênea vizinhança para
agregar.

In [ ]:
grau = pd.DataFrame([
    {"tabela": n, "aponta_para": G.out_degree(n), "apontada_por": G.in_degree(n)}
    for n in G.nodes
]).sort_values("apontada_por", ascending=False)
print(grau.head(10).to_string(index=False))

isoladas = [n for n in G.nodes if G.degree(n) == 0]
print(f"\ntabelas sem nenhuma chave estrangeira ({len(isoladas)}):")
print(f"  {isoladas}")
print("\nEssas entram no grafo como nós desconectados: a GNN não tem por onde")
print("propagar informação até elas, então só contribuem com features próprias.")


In [ ]:
fig, ax = plt.subplots(figsize=(15, 12))
pos = nx.spring_layout(G, k=0.9, iterations=120, seed=42)

perifericas = [n for n in G.nodes if n != TABELA_RAIZ]
nx.draw_networkx_nodes(G, pos, nodelist=perifericas, node_size=700,
                       node_color="#8fb8de", ax=ax)
nx.draw_networkx_nodes(G, pos, nodelist=[TABELA_RAIZ], node_size=2600,
                       node_color="#d95f4c", ax=ax)
nx.draw_networkx_edges(G, pos, alpha=0.35, arrowsize=9,
                       connectionstyle="arc3,rad=0.08", ax=ax)
nx.draw_networkx_labels(G, pos, font_size=7, ax=ax)

ax.set_title(f"Grafo relacional do CNES — {G.number_of_nodes()} tabelas, "
             f"{G.number_of_edges()} chaves estrangeiras\n"
             "derivado de docs/01-selecao-tabelas.md")
ax.axis("off")
plt.tight_layout()
plt.show()


## Verificação: nenhuma referência pendurada

O bug D-14 era uma chave estrangeira apontando para tabela não materializada. A
validação de `src/schema.py` já recusa isso no import, mas conferir aqui deixa a
propriedade visível no desenho.

In [ ]:
penduradas = [(o, c, d) for o, fk in schema.CNES_FKEY.items()
              for c, d in fk.items() if d not in set(schema.FACT_TABLES)]
print(f"chaves estrangeiras penduradas: {len(penduradas)}")
assert not penduradas, penduradas

componentes = list(nx.weakly_connected_components(G))
print(f"componentes fracamente conexos: {len(componentes)}")
print(f"maior componente: {max(len(c) for c in componentes)} tabelas")
print(f"\na raiz está no maior componente: "
      f"{TABELA_RAIZ in max(componentes, key=len)}")
